# Explorar el data lake del Sistema Inteligente de PQRS

Este cuaderno muestra **qué hay dentro del data lake** (RustFS, compatible con S3) y cómo se relaciona
con la **base de datos**: los buckets, las PQRS radicadas desde la web
(`pqrs-raw/radicaciones/<radicado>/`), sus partes (texto, foto y audio), el manifiesto, la
clasificación que guardó el harness, los datos sintéticos y los reportes.

**Antes de ejecutarlo:** el sistema debe estar encendido (`python scripts/sistema.py encender`).

**Solo lectura:** el cuaderno no modifica ni borra nada. Las contraseñas se leen del `.env` y no se
muestran en pantalla.

## 1. Librerías
Instala `boto3` (cliente S3) y `psycopg` (PostgreSQL) si no están. Solo tarda la primera vez.

In [ ]:
%pip install -q boto3 "psycopg[binary]" pandas

## 2. Conexión
Se leen las credenciales del archivo `.env` del proyecto. Desde el PC, el data lake responde en
`localhost:9000` y la base en `localhost:5432` (los puertos que publica Docker).

In [ ]:
import io
import json
from pathlib import Path

import boto3
import pandas as pd
import psycopg
from botocore.config import Config
from IPython.display import Audio, Image, JSON, Markdown, display

# Buscar el .env del proyecto (el cuaderno vive en notebooks/)
raiz = Path.cwd() if (Path.cwd() / ".env").exists() else Path.cwd().parent
env = {}
for linea in (raiz / ".env").read_text(encoding="utf-8").splitlines():
    linea = linea.strip()
    if linea and not linea.startswith("#") and "=" in linea:
        clave, valor = linea.split("=", 1)
        env[clave.strip()] = valor.strip().strip('"').strip("'")

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id=env["MINIO_ROOT_USER"],
    aws_secret_access_key=env["MINIO_ROOT_PASSWORD"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)
bd = psycopg.connect(host="localhost", port=5432, dbname="pqrs",
                     user=env["POSTGRES_USER"], password=env["POSTGRES_PASSWORD"])

def consulta(sql, params=None):
    """Ejecuta una consulta de solo lectura y devuelve un DataFrame."""
    with bd.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

def objetos(bucket, prefijo=""):
    """Lista todos los objetos de un bucket (con paginación, de 1.000 en 1.000)."""
    filas = []
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefijo):
        for o in pagina.get("Contents", []):
            filas.append({"clave": o["Key"], "kb": round(o["Size"] / 1024, 1),
                          "modificado": o["LastModified"].strftime("%Y-%m-%d %H:%M")})
    return pd.DataFrame(filas)

print("Conectado al data lake y a la base de datos.")

## 3. Los buckets
Un *bucket* es la "carpeta raíz" del data lake. Hay dos: `pqrs-raw` (originales) y `pqrs-reportes` (PDF).

In [ ]:
pd.DataFrame([{"bucket": b["Name"], "creado": b["CreationDate"].strftime("%Y-%m-%d %H:%M")}
              for b in s3.list_buckets()["Buckets"]])

## 4. Qué hay en `pqrs-raw`
Cuántos archivos y cuánto ocupan, por carpeta.

In [ ]:
raw = objetos("pqrs-raw")
raw["carpeta"] = raw["clave"].str.split("/").str[0] + "/"
raw.groupby("carpeta").agg(archivos=("clave", "count"), kb_total=("kb", "sum")).round(1)

## 5. Las PQRS radicadas desde la web: `pqrs-raw/radicaciones/<radicado>/`
Cada radicación es una carpeta con sus partes y un `manifiesto.json`, que la web escribe **de último**
para indicar que el paquete está completo.

In [ ]:
rad = objetos("pqrs-raw", "radicaciones/")
if rad.empty:
    print("Todavía no hay PQRS radicadas desde la web.")
else:
    rad["radicado"] = rad["clave"].str.split("/").str[1]
    rad["archivo"] = rad["clave"].str.split("/").str[2]
    display(rad.groupby("radicado").agg(archivos=("archivo", lambda a: ", ".join(sorted(a))),
                                         kb_total=("kb", "sum"),
                                         ultimo_cambio=("modificado", "max")))

## 6. Una radicación por dentro
Por defecto, la más reciente. Puedes escribir otro radicado en `RADICADO`.

In [ ]:
RADICADO = None   # p. ej. "PQRS-2026-000001"; None = la más reciente

if RADICADO is None and not rad.empty:
    RADICADO = rad.sort_values("modificado")["radicado"].iloc[-1]
carpeta = f"radicaciones/{RADICADO}"
manifiesto = json.loads(s3.get_object(Bucket="pqrs-raw", Key=f"{carpeta}/manifiesto.json")["Body"].read())
display(Markdown(f"### {RADICADO}  ·  `pqrs-raw/{carpeta}/manifiesto.json`"))
display(JSON(manifiesto, expanded=True))

### Las partes, tal como llegaron (originales sin transformar)

In [ ]:
TIPOS_AUDIO = {"wav": "audio/wav", "mp3": "audio/mpeg", "webm": "audio/webm",
               "ogg": "audio/ogg", "m4a": "audio/mp4", "opus": "audio/ogg"}

for parte in manifiesto["partes"]:
    clave = f"{carpeta}/{parte['archivo']}"
    datos = s3.get_object(Bucket="pqrs-raw", Key=clave)["Body"].read()
    display(Markdown(f"**{parte['tipo'].upper()}** · `pqrs-raw/{clave}` · {len(datos) / 1024:.1f} KB"))
    if parte["tipo"] == "texto":
        print(datos.decode("utf-8"))
    elif parte["tipo"] == "foto":
        display(Image(data=datos, width=520))
    elif parte["tipo"] == "audio":
        ext = parte["archivo"].rsplit(".", 1)[-1].lower()
        display(Audio(data=datos, mimetype=TIPOS_AUDIO.get(ext, "audio/wav")))

## 7. Lo que hizo el harness con esa radicación
La web registra la radicación (`radicaciones`) y el harness guarda el resultado (`pqrs_casos`) y el
texto y el tiempo de cada parte (`pqrs_partes`). Se unen por la ruta del archivo en el data lake.

In [ ]:
caso = consulta("""
    SELECT r.radicado, r.canal, c.tipo, c.tema, c.urgencia, c.area_destino,
           c.clasificador, c.segundos_clasificacion, g.estado
    FROM radicaciones r
    LEFT JOIN pqrs_casos c ON c.archivo_origen = 'pqrs-raw/' || r.archivo_origen
    LEFT JOIN gestion g     ON g.radicado = r.radicado
    WHERE r.radicado = %s""", (RADICADO,))
display(caso.T.rename(columns={0: "valor"}))

partes = consulta("""
    SELECT p.tipo, p.segundos, p.error, left(p.texto_extraido, 160) AS texto_extraido
    FROM pqrs_partes p
    JOIN radicaciones r ON p.archivo_origen = 'pqrs-raw/' || r.archivo_origen
    WHERE r.radicado = %s ORDER BY p.id""", (RADICADO,))
partes

## 8. Todas las radicaciones con su clasificación

In [ ]:
consulta("""
    SELECT r.radicado, r.canal, c.tipo, c.tema, c.urgencia, c.area_destino,
           coalesce(g.estado, CASE WHEN c.id IS NULL THEN 'en proceso' ELSE 'clasificada' END) AS estado,
           to_char(r.recibido_en, 'YYYY-MM-DD HH24:MI') AS recibido
    FROM radicaciones r
    LEFT JOIN pqrs_casos c ON c.archivo_origen = 'pqrs-raw/' || r.archivo_origen
    LEFT JOIN gestion g     ON g.radicado = r.radicado
    ORDER BY r.recibido_en DESC""")

## 9. Los datos sintéticos: `textos/`, `audios/`, `fotos/`
Los 30 casos de prueba (10 de cada canal), generados con semilla fija. Un ejemplo de cada uno:

In [ ]:
for prefijo in ["textos/", "audios/", "fotos/"]:
    lista = objetos("pqrs-raw", prefijo)
    if lista.empty:
        print(f"{prefijo}: vacío (genera los datos con synthetic_data/generar_datos.py)")
        continue
    clave = lista["clave"].iloc[0]
    datos = s3.get_object(Bucket="pqrs-raw", Key=clave)["Body"].read()
    display(Markdown(f"**{prefijo}** · {len(lista)} archivos · ejemplo: `pqrs-raw/{clave}`"))
    if prefijo == "textos/":
        print(datos.decode("utf-8")[:400])
    elif prefijo == "fotos/":
        display(Image(data=datos, width=420))
    else:
        display(Audio(data=datos, mimetype="audio/mpeg"))

## 10. Precisión del modelo sobre los datos sintéticos
Cuántos casos tienen clasificación y cómo se reparten (lo detallado está en `evaluar.py`).

In [ ]:
consulta("""
    SELECT canal, urgencia, count(*) AS casos
    FROM pqrs_casos WHERE archivo_origen NOT LIKE 'pqrs-raw/radicaciones/%%'
    GROUP BY canal, urgencia ORDER BY canal, urgencia""")

## 11. Los reportes: `pqrs-reportes`

In [ ]:
objetos("pqrs-reportes")

---
**Para la sustentación:** ejecuta todo con *Run All*, radica una PQRS nueva en la web y vuelve a
ejecutar desde la sección 5: aparece la carpeta nueva en el data lake, con sus archivos originales, y
en la sección 7 la clasificación que le dio el harness.

In [ ]:
bd.close()